# Extract eICU Time Series Data
Those features are considered:
* Heart Rate
* Respiratory Rate
* SpO2 (oxygen saturation, it's actually SpO2 although it's called SaO2, see official paper for details)
* Mean Blood Pressure

Didn't consider systolic pressure and diastolic pressure since they are used to calculate mean blood pressure, and they follow the same trend for most of the time

Pre-req: build eICU database and run `sql/patient_id.sql`

In [1]:
import sys
sys.path.append("../")
from helpers.loader import DataBaseLoader
load = DataBaseLoader(user='ansongeorge', password='', dbname='eicu', schema='eicu')

SQL_SCRIPT = """
SELECT
	vitalPeriodic.patientunitstayid,
	observationoffset,
	heartrate,
	CASE
		WHEN heartrate IS NULL THEN 1
		ELSE 0
	END AS heartrate_null,
	respiration,
	CASE
		WHEN respiration IS NULL THEN 1
		ELSE 0
	END AS respiration_null,
	sao2,
	CASE
		WHEN sao2 IS NULL THEN 1
		ELSE 0
	END AS sao2_null,
	systemicMean,
	CASE
		WHEN systemicMean IS NULL THEN 1
		ELSE 0
	END AS systemicmean_null
 
FROM vitalPeriodic

WHERE vitalPeriodic.patientunitstayid = {id}
AND
(SELECT MIN(observationoffset) FROM vitalPeriodic WHERE patientunitstayid = {id} GROUP BY patientunitstayid) <= {start_time} AND (SELECT MAX(observationoffset) FROM vitalPeriodic WHERE patientunitstayid = {id} GROUP BY patientunitstayid) >= {end_time}
AND
observationoffset <= {end_time} AND observationoffset >= {start_time}

ORDER BY observationoffset
"""

def extract(id, start_time, end_time):
    df = load.query(SQL_SCRIPT.format(id=id, end_time=end_time, start_time=start_time))
    return df

## Get patients and do train test split

In [2]:
unitstay_id = load['patient_id']
unitstay_id = unitstay_id.dropna()          # drop ones without labels, this is caused by the join operation
unitstay_id[['sepsis', 'ami', 'akf', 'heart_failure']] = unitstay_id[['sepsis', 'ami', 'akf', 'heart_failure']].astype(int)
unitstay_id.to_csv('../data/eicu-extract/unitstay_id.csv', index=False)
print(f"There are {len(unitstay_id)} patients in the database.")

There are 170586 patients in the database.


In [3]:
from helpers.utils import seed_everything
from sklearn.model_selection import train_test_split
seed_everything(2023)
X, y = unitstay_id.drop('hospital_expire_flag', axis=1), unitstay_id['hospital_expire_flag']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=2023, stratify=y)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


Seed set to 2023



check label and proportion of each group

In [4]:
print(f"Mortality rate in training set: {100*y_train.sum() / len(y_train):.2f}%\nMortality rate in testing set: {100*y_test.sum() / len(y_test):.2f}%")

Mortality rate in training set: 9.27%
Mortality rate in testing set: 9.27%


In [5]:
print(f"Training set size: {len(X_train)}\nTesting set size: {len(X_test)}")

Training set size: 127939
Testing set size: 42647


In [6]:
print(f"TRAINING\nSepsis: {100*len(X_train[X_train['sepsis'] == 1])/len(X_train):.2f}%\nAMI: {100*len(X_train[X_train['ami'] == 1])/len(X_train):.2f}%\nHeart Failure: {100*len(X_train[X_train['heart_failure'] == 1])/len(X_train):.2f}%\nAKF: {100*len(X_train[X_train['akf'] == 1])/len(X_train):.2f}%")
print(f"\nTESTING\nSepsis: {100*len(X_test[X_test['sepsis'] == 1])/len(X_test):.2f}%\nAMI: {100*len(X_test[X_test['ami'] == 1])/len(X_test):.2f}%\nHeart Failure: {100*len(X_test[X_test['heart_failure'] == 1])/len(X_test):.2f}%\nAKF: {100*len(X_test[X_test['akf'] == 1])/len(X_test):.2f}%")

TRAINING
Sepsis: 14.34%
AMI: 5.94%
Heart Failure: 9.78%
AKF: 11.18%

TESTING
Sepsis: 14.50%
AMI: 5.83%
Heart Failure: 9.65%
AKF: 11.42%


save

In [7]:
import pandas as pd
train = pd.concat([X_train, y_train], axis=1)
test = pd.concat([X_test, y_test], axis=1)
train.to_csv('../data/eicu-extract/TRAIN-unitstay_id.csv', index=False)
test.to_csv('../data/eicu-extract/TEST-unitstay_id.csv', index=False)

## Extract Time Series

In [8]:
import pandas as pd
train = pd.read_csv('../data/eicu-extract/TRAIN-unitstay_id.csv')
test = pd.read_csv('../data/eicu-extract/TEST-unitstay_id.csv')
train.head()

,patientunitstayid,sepsis,ami,heart_failure,akf,hospital_expire_flag
0,2121063,0,0,0,0,0
1,244478,0,0,0,0,0
2,2935120,0,0,0,0,0
3,3344999,0,0,0,0,0
4,422778,0,0,0,0,0


In [9]:
train['ami'].unique()

array([0, 1])

In [10]:
train['heart_failure'].unique()

array([0, 1])

In [11]:
train['akf'].unique()

array([0, 1])

In [13]:
start_time, end_time, timesteps = 60, 1440, 276

eligible = load.query(f"""
SELECT patientunitstayid
FROM eicu.vitalperiodic
GROUP BY patientunitstayid
HAVING MIN(observationoffset) <= {start_time}
   AND MAX(observationoffset) >= {end_time};
""")

eligible_ids = eligible["patientunitstayid"].astype(int).tolist()
len(eligible_ids)

114962

In [14]:
import numpy as np
from tqdm import tqdm

FEATURES = ["heartrate", "respiration", "sao2", "systemicmean"]

def fetch_window(ids_chunk, start_time, end_time):
    ids_sql = ",".join(map(str, ids_chunk))
    q = f"""
    SELECT patientunitstayid, observationoffset,
           heartrate, respiration, sao2, systemicmean
    FROM eicu.vitalperiodic
    WHERE patientunitstayid IN ({ids_sql})
      AND observationoffset BETWEEN {start_time} AND {end_time}
    ORDER BY patientunitstayid, observationoffset;
    """
    return load.query(q)

def lets_extract_fast(ids, labels_df, start_time, end_time, timesteps, chunk_size=2000):

    label_map = (
        labels_df.set_index("patientunitstayid")["hospital_expire_flag"]
        .dropna()
        .astype(int)
        .to_dict()
    )

    ids = [int(i) for i in ids if int(i) in label_map]

    results = []

    for i in tqdm(range(0, len(ids), chunk_size), desc="Chunks"):
        chunk = ids[i:i+chunk_size]
        vitals = fetch_window(chunk, start_time, end_time)
        if vitals.empty:
            continue

        for pid, g in vitals.groupby("patientunitstayid", sort=False):

            if len(g) < timesteps:
                continue

            off = g["observationoffset"].to_numpy()
            if len(off) < timesteps or not np.all(np.diff(off[:timesteps]) == 5):
                continue

            # (T, 4)
            x = g[FEATURES].to_numpy(dtype=float)[:timesteps, :]

            # null flags (T, 4)
            nulls = np.isnan(x).astype(float)

            # Build interleaved channels
            channels = []
            for j in range(len(FEATURES)):
                channels.append(x[:, j])       # vital
                channels.append(nulls[:, j])   # vital_null

            # convert to (8, T)
            channels = np.vstack(channels)

            # add label row (1, T)
            label_row = np.full((1, timesteps), label_map[int(pid)], dtype=float)

            # final shape (9, T)
            sample = np.vstack([channels, label_row])

            results.append(sample)

    return np.asarray(results)

### Training

extract with time interval of 5

In [15]:
filtered_train = train[train["patientunitstayid"].isin(eligible_ids)]

result3 = lets_extract_fast(
    eligible_ids,
    labels_df=filtered_train,      # must contain patientunitstayid + hospital_expire_flag
    start_time=60,
    end_time=1440,
    timesteps=276,
    chunk_size=2000
)

result3.shape

Chunks: 100%|██████████| 41/41 [33:25<00:00, 48.92s/it]


(46736, 9, 276)

In [16]:
import torch
torch.save(torch.tensor(result3.astype(np.float64)), '../data/eicu-extract/TRAIN-eicu_multiple_60_1440_276.pt')

### Testing

use same procedure as training set

In [17]:
filtered_test = test[test["patientunitstayid"].isin(eligible_ids)]

test_24hrs = lets_extract_fast(
    eligible_ids,
    labels_df=filtered_test,      # must contain patientunitstayid + hospital_expire_flag
    start_time=60,
    end_time=1440,
    timesteps=276,
    chunk_size=2000
)

test_24hrs.shape

Chunks: 100%|██████████| 14/14 [11:47<00:00, 50.57s/it]


(15717, 9, 276)

In [18]:
import torch
torch.save(torch.tensor(test_24hrs.astype(np.float64)), '../data/eicu-extract/TEST-eicu_multiple_60_1440_276.pt')